# Breast Cancer Detection: Logistic Regression
**Project:** Breast Cancer Detection Using Classical Machine Learning (WDBC dataset)
**Part:** Logistic Regression (compared later with KNN and SVM)

**Files needed in the same folder:** `train_unscaled.csv`, `test_unscaled.csv`
(these hold the shared stratified 80/20 split, `random_state=42`, so all models are compared on identical data).

Target: `diagnosis` (0 = benign, 1 = **malignant**, the positive class). Because a missed malignant tumour (false negative) is the costliest error, we focus on **recall**.

**Plan:** load data → baseline → stratified CV + hyperparameter tuning → test evaluation → confusion matrix / ROC → threshold analysis → coefficients → error analysis → save results.

In [1]:
import numpy as np, pandas as pd, json, os, warnings
os.environ["PYTHONWARNINGS"] = "ignore"   # silence sklearn deprecation noise (also in worker processes)
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, classification_report)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load the shared train/test split

In [2]:
train = pd.read_csv("train_unscaled.csv")
test  = pd.read_csv("test_unscaled.csv")

X_train, y_train = train.drop(columns="diagnosis"), train["diagnosis"]
X_test,  y_test  = test.drop(columns="diagnosis"),  test["diagnosis"]

print("Train:", X_train.shape, "| malignant share:", round(y_train.mean(), 3))
print("Test: ", X_test.shape,  "| malignant share:", round(y_test.mean(), 3))

Train: (455, 30) | malignant share: 0.374
Test:  (114, 30) | malignant share: 0.368


## 2. Model design
Logistic Regression is distance/scale-sensitive through regularisation and gradient-based fitting, so features must be standardised. The scaler is placed **inside a `Pipeline`**: during cross-validation it is refitted on each training fold only, so there is no data leakage.

In [3]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def evaluate(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy":    accuracy_score(y_true, y_pred),
        "Precision":   precision_score(y_true, y_pred),
        "Recall":      recall_score(y_true, y_pred),
        "F1":          f1_score(y_true, y_pred),
        "Specificity": tn / (tn + fp),
        "ROC-AUC":     roc_auc_score(y_true, y_prob),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

## 3. Baseline (default hyperparameters, 5-fold stratified CV on training data)

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc"}
res = cross_validate(baseline, X_train, y_train, cv=cv, scoring=scoring)
pd.DataFrame({k.replace("test_", ""): [v.mean(), v.std()]
              for k, v in res.items() if k.startswith("test_")},
             index=["mean", "std"]).round(4)

,accuracy,precision,recall,f1,roc_auc
mean,0.9736,0.9771,0.9529,0.9640,0.9958
std,0.0149,0.0280,0.0399,0.0207,0.0047


## 4. Hyperparameter tuning (GridSearchCV, stratified 5-fold)
Searched: regularisation strength `C`, penalty (L1 / L2), and `class_weight` (none vs balanced, relevant because malignant is the minority class). Refit metric is **F1** (balances precision and recall); recall and ROC-AUC are tracked too.

In [5]:
pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", LogisticRegression(solver="liblinear", max_iter=5000,
                                            random_state=RANDOM_STATE))])

param_grid = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l1", "l2"],
    "clf__class_weight": [None, "balanced"],
}

grid = GridSearchCV(pipe, param_grid, cv=cv, scoring=scoring, refit="f1",
                    n_jobs=-1, return_train_score=False)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1 :", round(grid.best_score_, 4))

cv_results = pd.DataFrame(grid.cv_results_)
cols = ["param_clf__C", "param_clf__penalty", "param_clf__class_weight",
        "mean_test_f1", "mean_test_recall", "mean_test_roc_auc"]
cv_results[cols].sort_values("mean_test_f1", ascending=False).head(8).round(4).reset_index(drop=True)

Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2'}
Best CV F1 : 0.9674


,param_clf__C,param_clf__penalty,param_clf__class_weight,mean_test_f1,mean_test_recall,mean_test_roc_auc
0,0.10,l2,balanced,0.9674,0.9588,0.9949
1,1.00,l1,balanced,0.9641,0.9529,0.9948
2,1.00,l2,NaN,0.9640,0.9529,0.9959
3,1.00,l1,NaN,0.9638,0.9471,0.9959
4,0.10,l2,NaN,0.9637,0.9412,0.9948
5,1.00,l2,balanced,0.9616,0.9588,0.9954
6,0.10,l1,NaN,0.9613,0.9529,0.9913
7,0.01,l2,balanced,0.9587,0.9529,0.9921


In [6]:
# Effect of C on CV performance (L2 penalty)
sub = cv_results[(cv_results.param_clf__penalty == "l2") & (cv_results.param_clf__class_weight.isna())]
plt.figure(figsize=(6, 3.5))
plt.semilogx(sub.param_clf__C.astype(float), sub.mean_test_f1, "o-", label="F1")
plt.semilogx(sub.param_clf__C.astype(float), sub.mean_test_recall, "s--", label="Recall")
plt.xlabel("C (inverse regularisation strength)"); plt.ylabel("CV score")
plt.title("Effect of C (L2, no class weight)"); plt.legend(); plt.tight_layout(); plt.show()

## 5. Final evaluation on the held-out test set (used once)

In [7]:
best = grid.best_estimator_
y_pred = best.predict(X_test)
y_prob = best.predict_proba(X_test)[:, 1]

metrics = evaluate(y_test, y_pred, y_prob)
pd.Series(metrics).to_frame("Logistic Regression").T.round(4)

,Accuracy,Precision,Recall,F1,Specificity,ROC-AUC,TN,FP,FN,TP
Logistic Regression,0.9912,1.0,0.9762,0.988,1.0,0.9977,72.0,0.0,1.0,41.0


In [8]:
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"], digits=4))

              precision    recall  f1-score   support

      Benign     0.9863    1.0000    0.9931        72
   Malignant     1.0000    0.9762    0.9880        42

    accuracy                         0.9912       114
   macro avg     0.9932    0.9881    0.9905       114
weighted avg     0.9913    0.9912    0.9912       114



## 6. Confusion matrix and ROC curve

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["Benign", "Malignant"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion matrix (test set)")

fpr, tpr, _ = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, label=f"Logistic Regression (AUC = {metrics['ROC-AUC']:.4f})")
ax[1].plot([0, 1], [0, 1], "k--", alpha=.4)
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate (recall)")
ax[1].set_title("ROC curve (test set)"); ax[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

## 7. Threshold analysis (reducing false negatives)
The default decision threshold is 0.5. Since a missed malignant case is the costly error, we check how lowering the threshold trades recall against precision. **Note:** the threshold should be chosen using cross-validated predictions on the training data, not the test set, to avoid tuning on test data. The test-set column is shown only to report the resulting effect.

In [10]:
from sklearn.model_selection import cross_val_predict

oof_prob = cross_val_predict(best, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    p_tr = (oof_prob >= t).astype(int)
    p_te = (y_prob >= t).astype(int)
    rows.append({"threshold": t,
                 "CV recall": recall_score(y_train, p_tr),
                 "CV precision": precision_score(y_train, p_tr),
                 "Test recall": recall_score(y_test, p_te),
                 "Test precision": precision_score(y_test, p_te),
                 "Test FN": int(((p_te == 0) & (y_test == 1)).sum()),
                 "Test FP": int(((p_te == 1) & (y_test == 0)).sum())})
pd.DataFrame(rows).round(4)

,threshold,CV recall,CV precision,Test recall,Test precision,Test FN,Test FP
0,0.1,0.9941,0.6898,1.0000,0.7500,0,14
1,0.2,0.9941,0.8164,1.0000,0.8571,0,7
2,0.3,0.9824,0.9027,0.9762,0.8913,1,5
3,0.4,0.9765,0.9432,0.9762,0.9318,1,3
4,0.5,0.9588,0.9760,0.9762,1.0000,1,0
5,0.6,0.9353,0.9876,0.9524,1.0000,2,0


## 8. Interpretability: coefficients and odds ratios
Features were standardised, so coefficients are directly comparable: a positive coefficient pushes towards **malignant**. `exp(coef)` is the odds multiplier for a one-standard-deviation increase in that feature. With an L1 penalty some coefficients may be exactly zero (feature selection). Caution: radius, perimeter and area are highly correlated (> 0.95), so individual coefficients among them are unstable; interpret them as a group.

In [11]:
coef = pd.Series(best.named_steps["clf"].coef_[0], index=X_train.columns)
coef_df = pd.DataFrame({"coef": coef, "odds_ratio": np.exp(coef), "abs": coef.abs()}) \
            .sort_values("abs", ascending=False).drop(columns="abs")

print("Non-zero coefficients:", int((coef != 0).sum()), "of", len(coef))
display(coef_df.head(10).round(4))

top = coef.reindex(coef.abs().sort_values(ascending=False).head(15).index)[::-1]
plt.figure(figsize=(7, 5))
plt.barh(top.index, top.values, color=np.where(top.values > 0, "#c0392b", "#2980b9"))
plt.axvline(0, color="k", lw=.8)
plt.title("Top 15 coefficients (red = towards malignant)")
plt.xlabel("Coefficient (standardised features)"); plt.tight_layout(); plt.show()

Non-zero coefficients: 30 of 30


,coef,odds_ratio
texture_worst,0.6235,1.8655
radius_worst,0.5081,1.6622
radius_se,0.4863,1.6263
area_worst,0.4641,1.5906
perimeter_worst,0.4617,1.5867
symmetry_worst,0.4521,1.5716
concave points_mean,0.4480,1.5652
concave points_worst,0.4430,1.5573
texture_mean,0.4235,1.5273
concavity_worst,0.3737,1.4531


## 9. Error analysis: which test samples were misclassified?

In [12]:
err = X_test.copy()
err["true"] = y_test.values
err["pred"] = y_pred
err["P(malignant)"] = y_prob.round(3)
errors = err[err.true != err.pred].copy()
errors["error_type"] = np.where((errors.true == 1) & (errors.pred == 0),
                                "False negative (missed malignant)", "False positive")
print(f"{len(errors)} misclassified of {len(err)} test samples")
errors[["error_type", "P(malignant)", "radius_mean", "area_mean", "concave points_mean",
        "radius_worst", "concave points_worst"]]

1 misclassified of 114 test samples


,error_type,P(malignant),radius_mean,area_mean,concave points_mean,radius_worst,concave points_worst
16,False negative (missed malignant),0.224,13.8,584.1,0.05069,16.57,0.1383


## 10. Save results for comparison with KNN and SVM

In [13]:
results = {"model": "Logistic Regression",
           "best_params": {k.replace("clf__", ""): v for k, v in grid.best_params_.items()},
           "cv_f1": round(grid.best_score_, 4),
           "test_metrics": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()}}
with open("results_logistic_regression.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(json.dumps(results, indent=2, default=str))

{
  "model": "Logistic Regression",
  "best_params": {
    "C": 0.1,
    "class_weight": "balanced",
    "penalty": "l2"
  },
  "cv_f1": 0.9674,
  "test_metrics": {
    "Accuracy": 0.9912,
    "Precision": 1.0,
    "Recall": 0.9762,
    "F1": 0.988,
    "Specificity": 1.0,
    "ROC-AUC": 0.9977,
    "TN": 72,
    "FP": 0,
    "FN": 1,
    "TP": 41
  }
}


## 11. Summary
- **Best hyperparameters** (5-fold stratified CV, refit on F1): `C = 0.1`, L2 penalty, `class_weight = "balanced"` (CV F1 = 0.967, CV recall = 0.959).
- **Test set (114 samples, 42 malignant):** accuracy 0.991, precision 1.000, recall 0.976, specificity 1.000, F1 0.988, ROC-AUC 0.998.
- **Errors:** one false negative (test row 16, a small malignant mass with radius_mean 13.8 and P(malignant) = 0.224) and no false positives.
- **Threshold:** lowering the threshold to 0.3 (chosen on out-of-fold training predictions) keeps recall at 0.976 with 5 false positives; 0.2 removes the false negative at the cost of 7 false positives.
- **Top features** (standardised coefficients): texture_worst, radius_worst, radius_se, area_worst, perimeter_worst. Radius, perimeter and area are highly correlated, so read them as a group.
- **Comparison:** `compare_all_models.py` re-runs this model alongside KNN, SVM, ID3, C4.5, CART, Random Forest and XGBoost on the same split and folds; Logistic Regression has the best test result there too.
